In [796]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
from sklearn.model_selection import train_test_split
from sklearn.ensemble import RandomForestRegressor, HistGradientBoostingRegressor
from sklearn.tree import DecisionTreeRegressor
import xgboost as xgb
from xgboost import XGBRegressor
from lightgbm import LGBMRegressor
from sklearn.metrics import mean_squared_error, mean_absolute_error, mean_absolute_percentage_error
from statsmodels.tsa.arima.model import ARIMA
import os
from sklearn.svm import SVR, LinearSVR
from sklearn.preprocessing import StandardScaler, MinMaxScaler, RobustScaler
from sklearn.linear_model import LinearRegression
from sklearn.neighbors import KNeighborsRegressor
import warnings

## load_data

In [797]:
def load_data(folder_path):
    dataframes = []
    for filename in os.listdir(folder_path):
        if filename.endswith('.csv'):
            file_path = os.path.join(folder_path, filename)
            df = pd.read_csv(file_path)
            category = os.path.splitext(filename)[0]
            df['Category'] = category
            dataframes.append(df)
    if dataframes:
        merged_data = pd.concat(dataframes, ignore_index=True)
        merged_data['Date'] = pd.to_datetime(merged_data['Date'])
    else:
        print("No CSV files found in the specified folder.")
        return pd.DataFrame()

    return merged_data

## get_date

In [798]:
def get_date(df, lag, target, is_forecast, last_known_value=None):
    df['Day'] = df.index.day
    df['day_of_week'] = df.index.dayofweek
    
    if is_forecast:
        for i in range(1, lag + 1):
            if last_known_value is None:
                df[f'lag_{i}'] = None
            else:
                df[f'lag_{i}'] = last_known_value
    else:
        for i in range(1, lag + 1):
            df[f'lag_{i}'] = df.groupby('Category')[target].shift(i)
    
    # Fill NaN values using backfill
    df.fillna(method='bfill', inplace=True)

    return df  # Return the DataFrame without dropping rows

## add_rolling_statistics

In [799]:
def add_rolling_statistics(df, window_size, target, is_forecast, last_known_mean=None, last_known_std=None):
    if is_forecast:
        # For forecast, fill with last known values
        df[f'rolling_mean_{window_size}'] = last_known_mean
        df[f'rolling_std_{window_size}'] = last_known_std
    else:
        # Calculate rolling mean and std with min_periods=1 to avoid NaNs
        rolling_mean = df.groupby('Category')[target].rolling(window=window_size, min_periods=1).mean()
        rolling_std = df.groupby('Category')[target].rolling(window=window_size, min_periods=1).std()
        
        # Assign the rolling statistics back to the DataFrame
        df[f'rolling_mean_{window_size}'] = rolling_mean.reset_index(level=0, drop=True)
        df[f'rolling_std_{window_size}'] = rolling_std.reset_index(level=0, drop=True)

    # Fill NaN values using bfill
    df[f'rolling_mean_{window_size}'].fillna(method='bfill', inplace=True)
    df[f'rolling_std_{window_size}'].fillna(method='bfill', inplace=True)

    return df

## add_fourier_features

In [800]:
def add_fourier_features(df, period):
    df['sin'] = np.sin(2 * np.pi * df.index.dayofyear / period)
    df['cos'] = np.cos(2 * np.pi * df.index.dayofyear / period)
    return df

## category_df

In [801]:
def category_df(df, category):
    return df[df['Category'] == category]

## split_data

In [802]:
def split_data(df, category, target):
    X = df.drop(columns=[target, 'Category'])
    y = df[target]
    X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2)
    return X_train, X_test, y_train, y_test, X, y

## scale

In [803]:
def scale(X_train, X_test):
    scaler = StandardScaler()
    X_train_scaled = scaler.fit_transform(X_train)
    X_test_scaled = scaler.transform(X_test)
    return pd.DataFrame(X_train_scaled, columns=X_train.columns), pd.DataFrame(X_test_scaled, columns=X_test.columns)

## train_model

In [804]:
def train_model(X_train, X_test, y_train, y_test, category):
    model = HistGradientBoostingRegressor()
    model.fit(X_train, y_train)
    y_pred = model.predict(X_test)
    mape = mean_absolute_percentage_error(y_test, y_pred)
    print(f'Mean Absolute Percentage Error for {category}: {mape}')
    if isinstance(X_train, pd.DataFrame):
        print("Trained model feature names:", X_train.columns.tolist())
    else:
        print("Trained model feature names: Not available (input is not a DataFrame)")
    return model

## make_df

In [805]:
def make_df(df, target, last_known_value=None, last_known_mean=None, last_known_std=None):
    dates = pd.date_range(start='2024-10-01', end='2024-12-31', freq='D')
    prediction_set = pd.DataFrame(index=dates)
    
    # Ensure the last known category is correctly assigned
    prediction_set['Category'] = df['Category'].iloc[-1]  # Assuming this is a single value
    
    # Initialize the target column with the last known value or None
    prediction_set[target] = last_known_value if last_known_value is not None else None
    prediction_set = pd.concat([df, prediction_set])
    # Get date and rolling statistics
    prediction_set = get_date(prediction_set, 3, target, True, last_known_value)
    # prediction_set = add_rolling_statistics(prediction_set, 2, target, True, last_known_mean, last_known_std)
    
    return prediction_set

## forecast

In [ ]:
def forecast(df, model, target):
    current_date = df.index[-1] + pd.Timedelta(days=1)
    
    # Create a new row for the next date
    new_row = pd.DataFrame(index=[current_date])
    last_known_value = df[target].iloc[-1]
    new_row[target] = last_known_value  # Initialize with the last known value
    
    # Concatenate the new row to the existing DataFrame
    prediction_set = pd.concat([df, new_row])
    
    # Update the prediction_set with new features for forecasting
    prediction_set = get_date(prediction_set, 3, target, True, last_known_value)
    
    # Prepare the features for prediction
    X_forecast = prediction_set.drop(columns=[target, 'Category']).iloc[-1:]  # Get the last row for prediction
    scaler = StandardScaler()
    X_forecast = scaler.fit_transform(X_forecast)
    
    # Make predictions
    y_forecast = model.predict(X_forecast)
    
    # Update the prediction_set with the forecasted value
    prediction_set.at[current_date, target] = y_forecast[0]
    
    
    return prediction_set

## pipeline

In [807]:
def pipeline(path, target):
    result = []
    df = load_data(path)
    warnings.simplefilter(action='ignore', category=FutureWarning)
    warnings.simplefilter(action='ignore', category=Warning)
    feature_columns = ['Day', 'day_of_week'] #, 'lag_1', 'lag_2', 'lag_3', 'rolling_mean_2', 'rolling_std_2']
    if target not in df.columns:
        raise KeyError(f"The target column '{target}' does not exist in the DataFrame.")
    
    if df[target].dtype == 'object':
        df[target] = df[target].replace({'\$':'', ',':'', '%':'', 'K':'000'}, regex=True)
        df[target] = df[target].apply(pd.to_numeric, errors='coerce')
    
    df = df.set_index('Date')   
    df = df[[target, 'Category']]
    
    
    df = get_date(df, 3, target, False)
    # df = add_rolling_statistics(df, 2, target, False)

    df['Category'] = df['Category'].astype('category')
    categories = df['Category'].unique()
    
    for category in categories:
        temp_df = category_df(df, category)
        
        
        X_train, X_test, y_train, y_test, X, y = split_data(temp_df, category, target)
        X_train, X_test = scale(X_train, X_test)
        
        model = train_model(X_train, X_test, y_train, y_test, category)
        
        # last_known_value = temp_df[target].iloc[-1]
        # last_known_mean = temp_df[f'rolling_mean_2'].iloc[-1]
        # last_known_std = temp_df[f'rolling_std_2'].iloc[-1]
        
        # forecast_df = make_df(temp_df, target) #, last_known_value, last_known_mean, last_known_std)
        # print(forecast_df.head())
        predictions = forecast(temp_df, model, target) #, last_known_value, last_known_mean, last_known_std, feature_columns)
        result.append(predictions)
    
    result_df = pd.concat(result, axis=1)
    return result_df, model, df

## Forecasting

In [808]:
forecast_uang, _, _ = pipeline(r'C:\Users\farel\OneDrive\Documents\GitHub\Arkavidia-9\Mata Uang', 'Close')

Mean Absolute Percentage Error for MYRUSD=X: 0.003320816173058366
Trained model feature names: ['Day', 'day_of_week', 'lag_1', 'lag_2', 'lag_3']
Mean Absolute Percentage Error for SGDUSD=X: 0.002812512028841909
Trained model feature names: ['Day', 'day_of_week', 'lag_1', 'lag_2', 'lag_3']
Mean Absolute Percentage Error for THBUSD=X: 0.004327804450156816
Trained model feature names: ['Day', 'day_of_week', 'lag_1', 'lag_2', 'lag_3']
Mean Absolute Percentage Error for USDIDR=X: 0.003836028598373803
Trained model feature names: ['Day', 'day_of_week', 'lag_1', 'lag_2', 'lag_3']


In [811]:
forecast_uang.columns

Index(['Close', 'Category', 'Day', 'day_of_week', 'lag_1', 'lag_2', 'lag_3',
       'Close', 'Category', 'Day', 'day_of_week', 'lag_1', 'lag_2', 'lag_3',
       'Close', 'Category', 'Day', 'day_of_week', 'lag_1', 'lag_2', 'lag_3',
       'Close', 'Category', 'Day', 'day_of_week', 'lag_1', 'lag_2', 'lag_3'],
      dtype='object')

In [809]:
forecast_uang.to_csv(r'C:\Users\farel\OneDrive\Documents\GitHub\Arkavidia-9\forecasted\uang.csv')

In [ ]:
def forecast(df, model, X, steps, category):
    # Get the last known values from the features
    last_known = X.iloc[-1].values.reshape(1, -1)
    forecast = []

    # Generate forecast for the specified number of steps
    for _ in range(steps):
        next_value = model.predict(last_known)
        forecast.append(next_value[0])
        
        # Update last_known with the new predicted value
        last_known = np.append(last_known[:, 1:], next_value).reshape(1, -1)

    # Create forecast dates starting from the last date in the original DataFrame
    forecast_dates = pd.date_range(start=df.index[-1] + pd.Timedelta(days=1), periods=steps)
    
    # Create a DataFrame for the forecasted values
    forecast_df = pd.DataFrame(forecast, index=forecast_dates, columns=[category])
    # Debugging: Print the shape of the forecast and forecast_df
    print(f"Forecast length: {len(forecast)}")
    print(f"Forecast DataFrame shape: {forecast_df.shape}")
    
    return forecast_df

In [ ]:
test.to_csv(r'C:\Users\farel\OneDrive\Documents\GitHub\Arkavidia-9\forecasted\uang.csv')

In [ ]:
test.describe()

,MYRUSD=X,SGDUSD=X,THBUSD=X,USDIDR=X
count,92.000000,92.000000,92.000000,92.000000
mean,0.223496,0.748578,0.028486,16424.453125
std,0.000259,0.001308,0.000000,3.080189
min,0.221041,0.736186,0.028486,16421.261719
25%,0.223523,0.748700,0.028486,16424.166016
50%,0.223523,0.748700,0.028486,16424.166016
75%,0.223523,0.748700,0.028486,16424.166016
max,0.223523,0.749284,0.028486,16453.535156


In [ ]:
# test.to_csv(r'C:\Users\farel\OneDrive\Documents\GitHub\Arkavidia-9\forecasted\uang.csv')